# BarqTrain Colab: Training + Inference

This notebook shows a full Colab workflow:
1. Install BarqTrain
2. Fine-tune a small Causal LM for a few steps
3. Apply `patch_model(...)`
4. Run inference

> Recommended runtime: **GPU** (`Runtime -> Change runtime type -> T4/A100`)
                


In [ ]:
%%bash
set -e
apt-get -y install python3-dev curl build-essential >/dev/null
if [ ! -x "$HOME/.cargo/bin/cargo" ]; then
  curl https://sh.rustup.rs -sSf | sh -s -- -y >/dev/null
fi
export PATH="$HOME/.cargo/bin:$PATH"
pip -qqq install --upgrade pip
pip -qqq install ninja packaging datasets accelerate peft trl
if [ ! -d /content/BarqTrain ]; then
  git clone https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
fi
cd /content/BarqTrain
pip install -q -e . --no-build-isolation
if [ -n "${COLAB_GPU:-}" ] || [ -d /usr/local/cuda ]; then
  BARQTRAIN_BUILD_CUDA=1 pip install -q -e . --no-build-isolation
fi
cargo --version


In [ ]:
%cd /content/BarqTrain

import importlib.util
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

os.environ['BARQTRAIN_REQUIRE_RUST'] = '1'
assert importlib.util.find_spec('barqtrain_rs'), 'barqtrain_rs did not build; re-run the install cell before importing barqtrain'

from barqtrain import PackedCausalLMDataCollator, patch_model

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('barqtrain_rs available:', bool(importlib.util.find_spec('barqtrain_rs')))
print('barqtrain_cuda available:', bool(importlib.util.find_spec('barqtrain_cuda')))
if torch.cuda.is_available():
    assert importlib.util.find_spec('barqtrain_cuda'), 'barqtrain_cuda did not build; re-run the install cell before training on GPU'


In [ ]:
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
MAX_LEN = 256

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.cuda()

# Apply BarqTrain patching
patch_model(model)

print('Model loaded and patched.')
                


In [ ]:
train_texts = [
    'BarqTrain accelerates LLM fine-tuning with fused kernels.',
    'Efficient training requires fast data loading and optimized attention.',
    'RMSNorm fusion can reduce memory traffic.',
    'This is a small demo dataset for Colab training.',
] * 64

raw_ds = Dataset.from_dict({'text': train_texts})

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

tokenized = raw_ds.map(tokenize, batched=True, remove_columns=['text'])
collator = PackedCausalLMDataCollator(
    max_length=MAX_LEN,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

args = TrainingArguments(
    output_dir='/content/barqtrain_colab_demo',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    max_steps=20,
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

trainer.train()


In [ ]:
prompt = "Write 3 tips to optimize LLM fine-tuning speed:" 
inputs = tokenizer(prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)

print(tokenizer.decode(out[0], skip_special_tokens=True))
                
